# 04. Baseline de regresión — consumo eléctrico del mes siguiente

## 1. Configuración y carga

Se detecta la raíz mediante el dataset oficial generado por Data Preparation. Todos los artefactos se guardan con rutas relativas construidas con `pathlib`.

In [1]:
from __future__ import annotations

from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

RANDOM_STATE = 42
DATASET_RELATIVE_PATH = Path("data") / "processed" / "dataset_ml.parquet"

def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / DATASET_RELATIVE_PATH).is_file():
            return candidate
    raise FileNotFoundError(
        f"No se encontró {DATASET_RELATIVE_PATH} desde {start} ni desde sus padres."
    )

ROOT = find_project_root()
DATASET_PATH = ROOT / DATASET_RELATIVE_PATH
OUTPUT_DIR = ROOT / "reports" / "model_outputs"
MODELS_DIR = ROOT / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

METRICS_PATH = OUTPUT_DIR / "metricas_baseline.csv"
DIST_METRICS_PATH = OUTPUT_DIR / "metricas_por_distribuidora.csv"
TARIFF_METRICS_PATH = OUTPUT_DIR / "metricas_por_tarifa.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "predicciones_test.csv"
SEARCH_PATH = OUTPUT_DIR / "resultados_validacion_arbol.csv"
MODEL_PATH = MODELS_DIR / "baseline_decision_tree.joblib"

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["font.size"] = 10

print(f"Raíz:   {ROOT}")
print(f"Dataset: {DATASET_PATH}")

Raíz:   C:\Users\rafae\Downloads\Hito1_1ACC0057_4773_GRUPO_08\Hito1_1ACC0057_4773_GRUPO_08
Dataset: C:\Users\rafae\Downloads\Hito1_1ACC0057_4773_GRUPO_08\Hito1_1ACC0057_4773_GRUPO_08\data\processed\dataset_ml.parquet


In [2]:
data = pl.read_parquet(DATASET_PATH)

required_columns = {
    "DISTRIBUIDORA", "UBIGEO", "TARIFA", "PERIODO", "TARGET_PERIODO",
    "MES", "consumo_total_mwh", "lag_1", "lag_2", "lag_3",
    "TARGET_CONSUMO_MWH", "SPLIT",
}
missing_columns = sorted(required_columns - set(data.columns))
if missing_columns:
    raise ValueError(f"Faltan columnas obligatorias: {missing_columns}")

split_overview = (
    data.group_by("SPLIT")
    .agg(
        pl.len().alias("observaciones"),
        pl.col("TARGET_PERIODO").min().alias("target_min"),
        pl.col("TARGET_PERIODO").max().alias("target_max"),
        pl.col("DISTRIBUIDORA").n_unique().alias("distribuidoras"),
        pl.col("UBIGEO").n_unique().alias("ubigeos"),
        pl.col("TARIFA").n_unique().alias("tarifas"),
    )
    .sort("target_min")
)

print(f"Dimensiones: {data.shape}")
print("Schema:")
print(data.schema)
print("Primeros registros:")
display(data.head())
print("Distribución y cobertura por split:")
display(split_overview)
print("Distribuidoras:", sorted(data["DISTRIBUIDORA"].unique().to_list()))
print("UBIGEO únicos:", data["UBIGEO"].n_unique())
print("Tarifas:", sorted(data["TARIFA"].unique().to_list()))

Dimensiones: (33174, 14)
Schema:
Schema({'DISTRIBUIDORA': String, 'UBIGEO': String, 'TARIFA': String, 'PERIODO': Int32, 'FECHA_PERIODO': Date, 'TARGET_PERIODO': Int32, 'ANIO': Int32, 'MES': Int8, 'consumo_total_mwh': Float64, 'lag_1': Float64, 'lag_2': Float64, 'lag_3': Float64, 'TARGET_CONSUMO_MWH': Float64, 'SPLIT': String})
Primeros registros:


DISTRIBUIDORA,UBIGEO,TARIFA,PERIODO,FECHA_PERIODO,TARGET_PERIODO,ANIO,MES,consumo_total_mwh,lag_1,lag_2,lag_3,TARGET_CONSUMO_MWH,SPLIT
str,str,str,i32,date,i32,i32,i8,f64,f64,f64,f64,f64,str
"""ENOSA""","""010509""","""BT5B""",202410,2024-10-01,202411,2024,10,0.0,0.0,0.0,0.0,0.0,"""TRAIN"""
"""ENOSA""","""010509""","""BT5B""",202411,2024-11-01,202412,2024,11,0.0,0.0,0.0,0.0,0.0,"""TRAIN"""
"""ENOSA""","""010509""","""BT5B""",202412,2024-12-01,202501,2024,12,0.0,0.0,0.0,0.0,0.0,"""VALIDATION"""
"""ENOSA""","""010509""","""BT5B""",202501,2025-01-01,202502,2025,1,0.0,0.0,0.0,0.0,0.0,"""VALIDATION"""
"""ENOSA""","""010509""","""BT5B""",202502,2025-02-01,202503,2025,2,0.0,0.0,0.0,0.0,0.0,"""VALIDATION"""


Distribución y cobertura por split:


SPLIT,observaciones,target_min,target_max,distribuidoras,ubigeos,tarifas
str,u32,i32,i32,u32,u32,u32
"""TRAIN""",19552,202304,202412,2,514,15
"""VALIDATION""",2936,202501,202503,2,471,14
"""TEST""",3934,202504,202507,2,476,14
"""FUTURE_HOLDOUT""",6752,202508,202608,1,200,14


Distribuidoras: ['ENOSA', 'HIDRANDINA']
UBIGEO únicos: 538
Tarifas: ['AT2', 'BT2', 'BT3', 'BT4', 'BT5A', 'BT5B', 'BT5C', 'BT5D', 'BT5E', 'BT6', 'BT8', 'MT2', 'MT2E', 'MT3', 'MT4']


## 2. Features, target y separación temporal

Features utilizadas:

- Categóricas: `DISTRIBUIDORA`, `UBIGEO`, `TARIFA`.
- Temporal: `MES`.
- Históricas: `consumo_total_mwh`, `lag_1`, `lag_2`, `lag_3`.

`consumo_total_mwh` es información disponible en `t`, por lo que es válida para estimar `t+1`.

In [3]:
CATEGORICAL_FEATURES = ["DISTRIBUIDORA", "UBIGEO", "TARIFA"]
NUMERIC_FEATURES = ["MES", "consumo_total_mwh", "lag_1", "lag_2", "lag_3"]
FEATURE_COLUMNS = CATEGORICAL_FEATURES + NUMERIC_FEATURES
TARGET = "TARGET_CONSUMO_MWH"
FORBIDDEN_FEATURES = {"TARGET_PERIODO", TARGET, "importe_total_miles", "IMPORTE", "SPLIT", "ANIO"}

assert not (set(FEATURE_COLUMNS) & FORBIDDEN_FEATURES)
assert sum(data.null_count().row(0)) == 0

train_pl = data.filter(pl.col("SPLIT") == "TRAIN")
val_pl = data.filter(pl.col("SPLIT") == "VALIDATION")
test_pl = data.filter(pl.col("SPLIT") == "TEST")
future_rows = data.filter(pl.col("SPLIT") == "FUTURE_HOLDOUT").height

train_max = train_pl["TARGET_PERIODO"].max()
val_min, val_max = val_pl["TARGET_PERIODO"].min(), val_pl["TARGET_PERIODO"].max()
test_min, test_max = test_pl["TARGET_PERIODO"].min(), test_pl["TARGET_PERIODO"].max()
assert train_max < val_min <= val_max < test_min <= test_max

def features_to_pandas(frame: pl.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(frame.select(FEATURE_COLUMNS).to_dict(as_series=False))

X_train = features_to_pandas(train_pl)
y_train = train_pl[TARGET].to_numpy()
X_val = features_to_pandas(val_pl)
y_val = val_pl[TARGET].to_numpy()

print("Features finales:", FEATURE_COLUMNS)
print(f"TRAIN: {len(train_pl):,} | VALIDATION: {len(val_pl):,} | TEST reservado: {len(test_pl):,}")
print(f"FUTURE_HOLDOUT reservado sin transformar: {future_rows:,}")
print(f"Orden temporal verificado: {train_max} < {val_min}–{val_max} < {test_min}–{test_max}")

Features finales: ['DISTRIBUIDORA', 'UBIGEO', 'TARIFA', 'MES', 'consumo_total_mwh', 'lag_1', 'lag_2', 'lag_3']
TRAIN: 19,552 | VALIDATION: 2,936 | TEST reservado: 3,934
FUTURE_HOLDOUT reservado sin transformar: 6,752
Orden temporal verificado: 202412 < 202501–202503 < 202504–202507


## 3. Métricas y referencia temporal

- **MAE:** error absoluto medio medido en MWh. 
- **RMSE:** también mide error en MWh, pero penaliza más los errores grandes.
- **R²:** indica qué parte de la variabilidad del consumo explica la predicción; no es un porcentaje de precisión.

No se usa MAPE porque existen consumos iguales o cercanos a cero.

In [4]:
def regression_metrics(y_true, y_pred) -> dict[str, float]:
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
    }

pred_reference_val = X_val["consumo_total_mwh"].to_numpy()
reference_val_metrics = regression_metrics(y_val, pred_reference_val)

print("Referencia de persistencia en VALIDATION:")
display(pd.DataFrame([reference_val_metrics], index=["Referencia temporal"]))

Referencia de persistencia en VALIDATION:


,MAE,RMSE,R2
Referencia temporal,14.646919,87.544245,0.99198


## 4. Pipeline y búsqueda pequeña de hiperparámetros

`OneHotEncoder(handle_unknown="ignore")` vive dentro del pipeline. En la búsqueda, cada pipeline se ajusta únicamente con TRAIN y Validation solo recibe `predict()`.

Se prueban 18 combinaciones comprensibles: seis profundidades y tres tamaños mínimos de hoja. La selección usa MAE de Validation; entre opciones dentro del 1% del mejor MAE se prefiere menor profundidad y, después, una hoja mínima mayor.

In [5]:
def build_pipeline(max_depth, min_samples_leaf: int) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            ("categoricas", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
            ("numericas", "passthrough", NUMERIC_FEATURES),
        ],
        remainder="drop",
        sparse_threshold=1.0,
    )
    tree = DecisionTreeRegressor(
        criterion="squared_error",
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=RANDOM_STATE,
    )
    return Pipeline([("preprocesamiento", preprocessor), ("modelo", tree)])

depths = [4, 6, 8, 10, 12, None]
leaf_sizes = [1, 10, 50]
configurations = [(depth, leaf) for depth in depths for leaf in leaf_sizes]
print(f"Configuraciones a evaluar: {len(configurations)}")

Configuraciones a evaluar: 18


In [6]:
search_rows = []
for max_depth, min_samples_leaf in configurations:
    pipeline = build_pipeline(max_depth, min_samples_leaf)
    pipeline.fit(X_train, y_train)
    pred_train = pipeline.predict(X_train)
    pred_val = pipeline.predict(X_val)
    train_metrics = regression_metrics(y_train, pred_train)
    val_metrics = regression_metrics(y_val, pred_val)
    search_rows.append({
        "max_depth": "None" if max_depth is None else str(max_depth),
        "min_samples_leaf": min_samples_leaf,
        "MAE_train": train_metrics["MAE"],
        "RMSE_train": train_metrics["RMSE"],
        "R2_train": train_metrics["R2"],
        "MAE_validation": val_metrics["MAE"],
        "RMSE_validation": val_metrics["RMSE"],
        "R2_validation": val_metrics["R2"],
        "generalization_gap_MAE": val_metrics["MAE"] - train_metrics["MAE"],
    })

search_results = pd.DataFrame(search_rows)
best_validation_mae = search_results["MAE_validation"].min()
eligible = search_results[
    search_results["MAE_validation"] <= best_validation_mae * 1.01
].copy()
eligible["depth_rank"] = eligible["max_depth"].map(lambda value: 999 if value == "None" else int(value))
selected_row = eligible.sort_values(
    ["depth_rank", "min_samples_leaf", "MAE_validation"],
    ascending=[True, False, True],
).iloc[0]

selected_depth = None if selected_row["max_depth"] == "None" else int(selected_row["max_depth"])
selected_leaf = int(selected_row["min_samples_leaf"])
search_results["seleccionado"] = (
    (search_results["max_depth"] == selected_row["max_depth"])
    & (search_results["min_samples_leaf"] == selected_leaf)
)
search_results = search_results.sort_values(
    ["MAE_validation", "max_depth", "min_samples_leaf"],
    ascending=[True, True, False],
).reset_index(drop=True)
search_results.to_csv(SEARCH_PATH, index=False)

print(f"Mejor MAE Validation absoluto: {best_validation_mae:.6f} MWh")
print(f"Configuración seleccionada: max_depth={selected_depth}, min_samples_leaf={selected_leaf}")
print("Resultados ordenados por MAE de Validation:")
display(search_results)

Mejor MAE Validation absoluto: 17.374729 MWh
Configuración seleccionada: max_depth=8, min_samples_leaf=10
Resultados ordenados por MAE de Validation:


,max_depth,min_samples_leaf,MAE_train,RMSE_train,R2_train,MAE_validation,RMSE_validation,R2_validation,generalization_gap_MAE,seleccionado
0,10,10,9.581853,84.743479,0.989667,17.374729,102.420292,0.989023,7.792876,False
1,8,10,10.081603,84.997853,0.989605,17.423408,102.283172,0.989052,7.341805,True
2,12,10,9.405448,84.695514,0.989679,17.498380,102.511322,0.989003,8.092932,False
3,None,10,9.283242,84.685054,0.989682,17.550832,102.517108,0.989002,8.267590,False
4,12,1,3.289177,8.920976,0.999885,18.807720,119.042999,0.985171,15.518543,False
5,6,10,12.955040,86.219147,0.989304,19.148434,102.699359,0.988963,6.193395,False
6,8,1,6.449958,21.248908,0.999350,19.960269,137.372821,0.980252,13.510310,False
7,10,1,4.618772,13.563512,0.999735,20.227903,141.027807,0.979187,15.609131,False
8,None,1,0.000345,0.024882,1.000000,20.796598,146.877027,0.977425,20.796253,False
9,6,1,11.074544,44.597354,0.997138,21.692143,138.108541,0.980040,10.617600,False


## 5. Comparación en Validation

La configuración ya fue elegida sin consultar TEST. Se vuelve a ajustar el pipeline seleccionado solo con TRAIN para compararlo honestamente contra la persistencia en Validation.

In [7]:
selected_pipeline = build_pipeline(selected_depth, selected_leaf)
selected_pipeline.fit(X_train, y_train)
pred_tree_val = selected_pipeline.predict(X_val)
tree_val_metrics = regression_metrics(y_val, pred_tree_val)
improvement_val_pct = (
    (reference_val_metrics["MAE"] - tree_val_metrics["MAE"])
    / reference_val_metrics["MAE"] * 100
)

validation_comparison = pd.DataFrame([
    {"Modelo": "Referencia temporal", **reference_val_metrics},
    {"Modelo": "Decision Tree", **tree_val_metrics},
])
print(f"Mejora MAE del árbol en Validation: {improvement_val_pct:.4f}%")
display(validation_comparison)

Mejora MAE del árbol en Validation: -18.9561%


,Modelo,MAE,RMSE,R2
0,Referencia temporal,14.646919,87.544245,0.991980
1,Decision Tree,17.423408,102.283172,0.989052


## 6. Análisis de overfitting

La configuración seleccionada (`max_depth=8`, `min_samples_leaf=10`) obtuvo MAE **10.081603 MWh** en TRAIN y **17.423408 MWh** en VALIDATION. La brecha absoluta fue **7.341805 MWh** (72.82% respecto al MAE de TRAIN).

El árbol sin límite de profundidad y con hojas de una observación obtuvo MAE **0.000345 MWh** en TRAIN y **20.796598 MWh** en VALIDATION. 
Esta comparación nos muestra que aumentar libremente la complejidad reduce mucho el error de entrenamiento, pero no garantiza una mejora equivalente fuera de muestra. Por ello se conserva la configuración regularizada elegida mediante Validation y la regla de simplicidad.

## 7. Reentrenamiento final y evaluación en TEST

Los hiperparámetros ya están fijados. Se crea un pipeline nuevo y se ajusta, incluido su encoder, únicamente con TRAIN + VALIDATION. Después se materializa TEST y se utiliza solo para `predict()`. 

In [8]:
train_val_pl = pl.concat([train_pl, val_pl])
X_train_val = features_to_pandas(train_val_pl)
y_train_val = train_val_pl[TARGET].to_numpy()

# TEST se materializa únicamente después de seleccionar los hiperparámetros.
X_test = features_to_pandas(test_pl)
y_test = test_pl[TARGET].to_numpy()
pred_reference_test = X_test["consumo_total_mwh"].to_numpy()

final_pipeline = build_pipeline(selected_depth, selected_leaf)
final_pipeline.fit(X_train_val, y_train_val)
pred_tree_test = final_pipeline.predict(X_test)

reference_test_metrics = regression_metrics(y_test, pred_reference_test)
tree_test_metrics = regression_metrics(y_test, pred_tree_test)
improvement_test_pct = (
    (reference_test_metrics["MAE"] - tree_test_metrics["MAE"])
    / reference_test_metrics["MAE"] * 100
)

metrics_baseline = pd.DataFrame([
    {"CONJUNTO": "VALIDATION", "MODELO": "Referencia temporal", **reference_val_metrics, "MEJORA_MAE_PCT": 0.0},
    {"CONJUNTO": "VALIDATION", "MODELO": "Decision Tree", **tree_val_metrics, "MEJORA_MAE_PCT": improvement_val_pct},
    {"CONJUNTO": "TEST", "MODELO": "Referencia temporal", **reference_test_metrics, "MEJORA_MAE_PCT": 0.0},
    {"CONJUNTO": "TEST", "MODELO": "Decision Tree", **tree_test_metrics, "MEJORA_MAE_PCT": improvement_test_pct},
])

print(f"Pipeline final ajustado con TRAIN + VALIDATION: {len(train_val_pl):,} filas")
print(f"TEST usado solo para predicción: {len(test_pl):,} filas")
display(metrics_baseline)

Pipeline final ajustado con TRAIN + VALIDATION: 22,488 filas
TEST usado solo para predicción: 3,934 filas


,CONJUNTO,MODELO,MAE,RMSE,R2,MEJORA_MAE_PCT
0,VALIDATION,Referencia temporal,14.646919,87.544245,0.991980,0.000000
1,VALIDATION,Decision Tree,17.423408,102.283172,0.989052,-18.956133
2,TEST,Referencia temporal,10.194128,58.970655,0.996104,0.000000
3,TEST,Decision Tree,12.715930,66.128861,0.995101,-24.737787


## 8. Resultados por distribuidora y tarifa

In [9]:
test_metadata = pd.DataFrame(
    test_pl.select(["DISTRIBUIDORA", "UBIGEO", "TARIFA", "PERIODO", "TARGET_PERIODO"]).to_dict(as_series=False)
)

distributor_rows = []
for distributor in sorted(test_metadata["DISTRIBUIDORA"].unique()):
    mask = test_metadata["DISTRIBUIDORA"].to_numpy() == distributor
    ref_metrics = regression_metrics(y_test[mask], pred_reference_test[mask])
    tree_metrics = regression_metrics(y_test[mask], pred_tree_test[mask])
    improvement = (ref_metrics["MAE"] - tree_metrics["MAE"]) / ref_metrics["MAE"] * 100
    distributor_rows.extend([
        {"DISTRIBUIDORA": distributor, "MODELO": "Referencia temporal", **ref_metrics, "MEJORA_MAE_PCT": 0.0},
        {"DISTRIBUIDORA": distributor, "MODELO": "Decision Tree", **tree_metrics, "MEJORA_MAE_PCT": improvement},
    ])
distributor_metrics = pd.DataFrame(distributor_rows)

tariff_rows = []
for tariff in sorted(test_metadata["TARIFA"].unique()):
    mask = test_metadata["TARIFA"].to_numpy() == tariff
    tariff_rows.append({
        "TARIFA": tariff,
        "cantidad_observaciones": int(mask.sum()),
        "MAE_referencia": float(mean_absolute_error(y_test[mask], pred_reference_test[mask])),
        "MAE_decision_tree": float(mean_absolute_error(y_test[mask], pred_tree_test[mask])),
    })
tariff_metrics = pd.DataFrame(tariff_rows).sort_values("cantidad_observaciones", ascending=False)

print("Métricas de TEST por distribuidora:")
display(distributor_metrics)
print("Resumen de TEST por tarifa:")
display(tariff_metrics)

Métricas de TEST por distribuidora:


,DISTRIBUIDORA,MODELO,MAE,RMSE,R2,MEJORA_MAE_PCT
0,ENOSA,Referencia temporal,11.569976,62.043565,0.992851,0.000000
1,ENOSA,Decision Tree,14.091222,75.734504,0.989348,-21.791276
2,HIDRANDINA,Referencia temporal,8.663109,55.350995,0.997615,0.000000
3,HIDRANDINA,Decision Tree,11.185530,53.447644,0.997776,-29.116806


Resumen de TEST por tarifa:


,TARIFA,cantidad_observaciones,MAE_referencia,MAE_decision_tree
4,BT5B,1971,13.327585,16.428713
8,BT6,389,0.291825,1.276135
12,MT3,380,21.670423,28.235418
10,MT2,270,9.613297,10.357671
13,MT4,227,3.979427,4.671521
1,BT3,160,4.927813,6.230880
6,BT5D,153,2.588693,3.304934
2,BT4,120,1.617583,2.045372
0,BT2,72,1.333472,1.596513
9,BT8,64,0.002813,0.479502


## 9. Visualizaciones de TEST

In [10]:
# Gráfico 1: real vs predicho, escala original y log1p.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, pred_tree_test, alpha=0.35, s=12, color="#2563eb")
limit = max(float(np.max(y_test)), float(np.max(pred_tree_test)))
axes[0].plot([0, limit], [0, limit], "--", color="#dc2626", linewidth=1.5)
axes[0].set_title("Decision Tree: real vs predicho (TEST)")
axes[0].set_xlabel("Consumo real (MWh)")
axes[0].set_ylabel("Consumo predicho (MWh)")

log_real = np.log1p(y_test)
log_pred = np.log1p(np.clip(pred_tree_test, 0, None))
log_limit = max(float(np.max(log_real)), float(np.max(log_pred)))
axes[1].scatter(log_real, log_pred, alpha=0.35, s=12, color="#7c3aed")
axes[1].plot([0, log_limit], [0, log_limit], "--", color="#dc2626", linewidth=1.5)
axes[1].set_title("Vista log1p")
axes[1].set_xlabel("log1p(consumo real)")
axes[1].set_ylabel("log1p(consumo predicho)")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "real_vs_predicho_test.png", dpi=150, bbox_inches="tight")
plt.show()

# Gráfico 2: totales mensuales reales y predichos.
temporal_plot = test_metadata[["TARGET_PERIODO"]].copy()
temporal_plot["Real"] = y_test
temporal_plot["Decision Tree"] = pred_tree_test
temporal_plot["Referencia temporal"] = pred_reference_test
temporal_plot = temporal_plot.groupby("TARGET_PERIODO", as_index=False).sum()

fig, ax = plt.subplots(figsize=(9, 5))
for column, color in [("Real", "#111827"), ("Decision Tree", "#2563eb"), ("Referencia temporal", "#f59e0b")]:
    ax.plot(temporal_plot["TARGET_PERIODO"].astype(str), temporal_plot[column], marker="o", label=column, color=color)
ax.set_title("Consumo total real y predicho por periodo — TEST")
ax.set_xlabel("TARGET_PERIODO")
ax.set_ylabel("Consumo total (MWh)")
ax.legend()
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "consumo_por_periodo_test.png", dpi=150, bbox_inches="tight")
plt.show()

# Gráfico 3: comparación global de MAE.
fig, ax = plt.subplots(figsize=(7, 4))
labels = ["Referencia temporal", "Decision Tree"]
values = [reference_test_metrics["MAE"], tree_test_metrics["MAE"]]
bars = ax.bar(labels, values, color=["#f59e0b", "#2563eb"])
ax.bar_label(bars, fmt="%.3f")
ax.set_title("MAE global en TEST")
ax.set_ylabel("MAE (MWh)")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "comparacion_mae_test.png", dpi=150, bbox_inches="tight")
plt.show()

# Gráfico 4: MAE por distribuidora.
dist_pivot = distributor_metrics.pivot(index="DISTRIBUIDORA", columns="MODELO", values="MAE")
dist_pivot = dist_pivot[["Referencia temporal", "Decision Tree"]]
ax = dist_pivot.plot(kind="bar", figsize=(8, 4), color=["#f59e0b", "#2563eb"], rot=0)
ax.set_title("MAE por distribuidora — TEST")
ax.set_ylabel("MAE (MWh)")
ax.legend(title="Modelo")
plt.tight_layout()
ax.figure.savefig(OUTPUT_DIR / "mae_por_distribuidora_test.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\rafae\AppData\Local\Temp\ipykernel_22816\1340807709.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\rafae\AppData\Local\Temp\ipykernel_22816\1340807709.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\rafae\AppData\Local\Temp\ipykernel_22816\1340807709.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\rafae\AppData\Local\Temp\ipykernel_22816\1340807709.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Exportación y verificación del modelo

Se exportan métricas, predicciones, búsqueda completa y el pipeline final. La recarga con joblib debe reproducir exactamente las predicciones de TEST.

In [11]:
predictions_test = test_metadata.copy()
predictions_test[TARGET] = y_test
predictions_test["PRED_REFERENCIA"] = pred_reference_test
predictions_test["PRED_DECISION_TREE"] = pred_tree_test
predictions_test["ERROR_ABS_REFERENCIA"] = np.abs(y_test - pred_reference_test)
predictions_test["ERROR_ABS_DECISION_TREE"] = np.abs(y_test - pred_tree_test)

metrics_baseline.to_csv(METRICS_PATH, index=False)
distributor_metrics.to_csv(DIST_METRICS_PATH, index=False)
tariff_metrics.to_csv(TARIFF_METRICS_PATH, index=False)
predictions_test.to_csv(PREDICTIONS_PATH, index=False)
joblib.dump(final_pipeline, MODEL_PATH)

reloaded_pipeline = joblib.load(MODEL_PATH)
reloaded_predictions = reloaded_pipeline.predict(X_test)
assert np.allclose(reloaded_predictions, pred_tree_test)
assert future_rows == 6_752

unseen_categories = {
    column: sorted(set(X_test[column]) - set(X_train_val[column]))
    for column in CATEGORICAL_FEATURES
}

print("Archivos exportados:")
for path in [METRICS_PATH, DIST_METRICS_PATH, TARIFF_METRICS_PATH, PREDICTIONS_PATH, SEARCH_PATH, MODEL_PATH]:
    print(f"- {path}")
print("Pipeline recargado correctamente; predicciones reproducidas.")
print("Categorías presentes en TEST pero no vistas en TRAIN + VALIDATION:")
print({column: len(values) for column, values in unseen_categories.items()})

Archivos exportados:
- C:\Users\rafae\Downloads\Hito1_1ACC0057_4773_GRUPO_08\Hito1_1ACC0057_4773_GRUPO_08\reports\model_outputs\metricas_baseline.csv
- C:\Users\rafae\Downloads\Hito1_1ACC0057_4773_GRUPO_08\Hito1_1ACC0057_4773_GRUPO_08\reports\model_outputs\metricas_por_distribuidora.csv
- C:\Users\rafae\Downloads\Hito1_1ACC0057_4773_GRUPO_08\Hito1_1ACC0057_4773_GRUPO_08\reports\model_outputs\metricas_por_tarifa.csv
- C:\Users\rafae\Downloads\Hito1_1ACC0057_4773_GRUPO_08\Hito1_1ACC0057_4773_GRUPO_08\reports\model_outputs\predicciones_test.csv
- C:\Users\rafae\Downloads\Hito1_1ACC0057_4773_GRUPO_08\Hito1_1ACC0057_4773_GRUPO_08\reports\model_outputs\resultados_validacion_arbol.csv
- C:\Users\rafae\Downloads\Hito1_1ACC0057_4773_GRUPO_08\Hito1_1ACC0057_4773_GRUPO_08\models\baseline_decision_tree.joblib
Pipeline recargado correctamente; predicciones reproducidas.
Categorías presentes en TEST pero no vistas en TRAIN + VALIDATION:
{'DISTRIBUIDORA': 0, 'UBIGEO': 8, 'TARIFA': 0}


## 11. FUTURE_HOLDOUT reservado

Las 6,752 observaciones de FUTURE_HOLDOUT no participaron en ajuste, selección, predicción ni cálculo de métricas. 

Las dejamos intactas para una futura evaluación temporal extendida sobre el dataset de ENOSA, asi tambien evitando el desbalanceo de clases.

## 12. Conclusiones

- Se evaluaron 18 configuraciones del árbol de decisión y se seleccionó max_depth=8 y min_samples_leaf=10, priorizando un buen equilibrio entre desempeño y complejidad.
- La referencia temporal obtuvo mejores resultados que el Decision Tree tanto en Validation como en Test. En Test, el MAE fue de 10.19 MWh para la referencia y 12.72 MWh para el árbol.
- El árbol presentó cierto sobreajuste, aunque la regularización redujo considerablemente este problema frente a un árbol sin restricciones.
- El comportamiento fue similar en ENOSA e Hidrandina: la referencia temporal mantuvo menor MAE en ambas distribuidoras.
- Estos resultados confirman que la persistencia mensual es un baseline fuerte. Como siguiente paso se propone evaluar nuevas variables y modelos más robustos, manteniendo los mismos cortes temporales y reservando FUTURE_HOLDOUT para una evaluación posterior.